# 03 · Neural Network Basics

In plain English, a neural network is a machine that **multiplies numbers, adds them up, bends the result a little, and repeats** — and a giant language model is just an enormous version of that same machine. In this notebook we build the machine from scratch, one piece at a time, using nothing but NumPy. We start with a single artificial "neuron," stack a few of them into a tiny network, measure how wrong it is, and then learn the one trick that lets it improve: **gradient descent**. By the end you'll train a real (tiny) neuron on a toy dataset and watch its error shrink step by step.

No heavy math. We stick to multiply and add, and we'll explain the scary-sounding "chain rule" as nothing more than **assigning blame**.

## What you'll learn

- **A single neuron**: weighted sum of inputs + bias, then an activation — built by hand in NumPy.
- **Activation functions** (sigmoid, ReLU, tanh): what they do and **why nonlinearity is essential** (a stack of plain linear layers secretly collapses into one linear layer).
- **Layers and a small network**: input → hidden → output, with a 2-layer forward pass using `np.dot`.
- **Loss**: a single number that measures how wrong the network is (mean squared error and binary cross-entropy).
- **Gradients & backpropagation, intuitively**: a gradient is "which way reduces the loss," and backprop is "passing blame backward through the layers."
- **A hands-on demo**: train one neuron with manual gradient descent and watch the loss go *down*, then plot the loss curve.

## Why this matters for fine-tuning

A large language model (LLM) is **a neural network** — the very same idea you're about to build, just scaled to billions of weights and many layers. Everything in this notebook is the foundation underneath fine-tuning:

- The model turns inputs into outputs through stacked layers of **weighted sums + activations** — exactly what we implement here.
- Training (and fine-tuning) means computing a **loss** (how wrong the model is on *your* data) and then nudging the weights to make that loss smaller.
- The nudging is **gradient descent driven by backpropagation** — the blame-passing trick we'll do by hand on one neuron.

So fine-tuning is not magic: it's **continuing to nudge an already-trained network's weights with gradient descent, using your examples**. Understand the tiny version and the billion-parameter version stops being mysterious.

## Setup

Run the cell below once. The `%pip install` line is **commented out** — uncomment it on Google Colab or a fresh environment that doesn't already have NumPy and Matplotlib. We use **only NumPy** for the math (no PyTorch yet — that's the next notebook) and Matplotlib for a couple of plots.

In [ ]:
# Uncomment the next line on Colab or a fresh environment:
# %pip install numpy matplotlib

import numpy as np                 # NumPy: fast number arrays and math
import matplotlib.pyplot as plt    # Matplotlib: simple plots

np.random.seed(0)   # makes random numbers repeatable so your results match ours
print("Imports OK")  # -> Imports OK

## 1. A single neuron

A neuron does three small things, in order:

1. **Multiply** each input by its own **weight** (a weight says "how much do I care about this input?").
2. **Add** all those products together, plus one extra number called the **bias** (a constant nudge).
3. **Activate**: pass that sum through a simple bending function (we'll meet several in a moment).

That's it. Step 1 + step 2 together are just a **weighted sum**. If your inputs are `x = [x1, x2]` and weights are `w = [w1, w2]`, the weighted sum is:

`z = w1*x1 + w2*x2 + b`

In NumPy, `np.dot(w, x)` does the "multiply-pairs-then-add-them-up" part in one call.

In [ ]:
# One neuron with 2 inputs.
x = np.array([1.0, 2.0])     # the inputs (e.g. two features of one example)
w = np.array([0.5, -1.0])    # one weight per input
b = 0.5                      # the bias (a single number)

# Step 1 + 2: weighted sum. np.dot multiplies pairs and adds them up:
#   0.5*1.0 + (-1.0)*2.0 = 0.5 - 2.0 = -1.5, then + bias 0.5 = -1.0
z = np.dot(w, x) + b
print("weighted sum z =", z)   # -> weighted sum z = -1.0

# Step 3: activation. We'll use sigmoid, which squashes any number into (0, 1).
def sigmoid(v):
    return 1.0 / (1.0 + np.exp(-v))

output = sigmoid(z)
print("neuron output =", round(float(output), 4))  # -> about 0.2689

**What this does:** It implements a complete neuron. `np.dot(w, x)` computes `w1*x1 + w2*x2`, we add the bias `b` to get the raw score `z`, and then `sigmoid(z)` bends that score into a number between 0 and 1 — handy when the output should look like a probability ("how confident is the neuron?"). Change the weights and watch the output move.

### ✏️ Exercise

**Task:** Write a function `neuron(x, w, b)` that returns the sigmoid-activated output of a neuron for any inputs `x`, weights `w`, and bias `b`. Test it with `x = [2.0, 3.0]`, `w = [1.0, 1.0]`, `b = -4.0`.

**Hint:** You only need one line inside the function: combine `np.dot`, `+ b`, and the `sigmoid` you already defined.

In [ ]:
# Sample solution
def neuron(x, w, b):
    return sigmoid(np.dot(w, x) + b)

result = neuron(np.array([2.0, 3.0]), np.array([1.0, 1.0]), -4.0)
print("neuron output =", round(float(result), 4))
# z = 1*2 + 1*3 - 4 = 1.0, sigmoid(1.0) -> about 0.7311

## 2. Activation functions (and why we need them)

The activation is the "bend" at the end of the neuron. Three common ones:

- **sigmoid**: squashes any number into **(0, 1)**. Great for probabilities.
- **tanh**: squashes into **(-1, 1)** — like sigmoid but centered at zero.
- **ReLU** ("rectified linear unit"): dead simple — `max(0, z)`. Negatives become 0, positives pass through unchanged. It's the workhorse inside most modern networks (including the ones in LLMs) because it's fast and trains well.

Let's implement all three and plot them.

In [ ]:
# The three activations, each taking a NumPy array and returning one.
def sigmoid(v):
    return 1.0 / (1.0 + np.exp(-v))

def tanh(v):
    return np.tanh(v)              # NumPy already has tanh built in

def relu(v):
    return np.maximum(0.0, v)     # element-wise max against 0

zs = np.linspace(-6, 6, 200)      # 200 evenly spaced inputs from -6 to 6

plt.figure(figsize=(7, 4))
plt.plot(zs, sigmoid(zs), label="sigmoid")
plt.plot(zs, tanh(zs), label="tanh")
plt.plot(zs, relu(zs), label="ReLU")
plt.axhline(0, color="gray", linewidth=0.5)
plt.axvline(0, color="gray", linewidth=0.5)
plt.title("Three activation functions")
plt.xlabel("input z"); plt.ylabel("output")
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()
# -> A plot: sigmoid is an S-curve in (0,1), tanh an S-curve in (-1,1),
#    ReLU is flat 0 for negatives then a straight diagonal line for positives.

**What this does:** It defines `sigmoid`, `tanh`, and `relu` so they work on whole arrays at once, evaluates each across 200 inputs, and plots them. Notice every curve is **bent** — none of them is a straight line. That bend is the whole point, as the next part shows.

### Why nonlinearity is essential

Here's the key idea. Suppose we **skip** the activations and just stack plain weighted sums (linear layers). A linear layer is `output = W·x + b`. Stack two of them:

- Layer 1: `h = W1·x + b1`
- Layer 2: `y = W2·h + b2 = W2·(W1·x + b1) + b2`

Multiply it out and you get `y = (W2·W1)·x + (W2·b1 + b2)`. That's **still just one weighted sum** of `x` — a single linear layer in disguise! No matter how many linear layers you stack, they collapse into one. To learn curvy, complicated patterns you must insert a **nonlinear** bend (an activation) between layers. Let's prove the collapse numerically.

In [ ]:
# Prove that two stacked LINEAR layers collapse into one linear layer.
x = np.array([1.0, 2.0, 3.0])

W1 = np.random.randn(4, 3)   # layer 1: maps 3 inputs -> 4 hidden values
b1 = np.random.randn(4)
W2 = np.random.randn(2, 4)   # layer 2: maps 4 hidden -> 2 outputs
b2 = np.random.randn(2)

# Path A: run the two linear layers one after another (NO activation between).
h = W1 @ x + b1              # "@" is matrix-multiply, same as np.dot here
y_two_layers = W2 @ h + b2

# Path B: collapse them into a SINGLE equivalent linear layer.
W_combined = W2 @ W1                 # one combined weight matrix
b_combined = W2 @ b1 + b2            # one combined bias
y_one_layer = W_combined @ x + b_combined

print("two linear layers:", np.round(y_two_layers, 5))
print("one combined layer:", np.round(y_one_layer, 5))
print("identical?", np.allclose(y_two_layers, y_one_layer))  # -> identical? True

**What this does:** It runs two linear layers (no activation) and then builds a single layer (`W2·W1`, `W2·b1 + b2`) that produces the **exact same numbers**. `np.allclose` confirms they match. That's the collapse: without a nonlinear activation in between, depth buys you nothing. Activations are what make deep networks actually *deep*.

### ✏️ Exercise

**Task:** Implement **Leaky ReLU**: it returns `z` when `z > 0`, but instead of a flat 0 for negatives it returns a small slope, `0.01 * z`. Apply it to the array `np.array([-3.0, -0.5, 0.0, 2.0])` and print the result.

**Hint:** `np.where(condition, value_if_true, value_if_false)` handles this in one line.

In [ ]:
# Sample solution
def leaky_relu(v):
    return np.where(v > 0, v, 0.01 * v)

print(leaky_relu(np.array([-3.0, -0.5, 0.0, 2.0])))
# -> [-0.03  -0.005  0.    2.   ]

## 3. From one neuron to a tiny network

A **layer** is just several neurons side by side, all reading the same inputs. We store a whole layer's weights in a 2D array (a matrix): one row of weights per neuron. Then `np.dot` (or `@`) computes every neuron's weighted sum **at once**.

We'll build a tiny 2-layer network:

- **Input**: 3 numbers.
- **Hidden layer**: 4 neurons, each reading all 3 inputs, with a ReLU activation.
- **Output layer**: 1 neuron reading the 4 hidden values, with a sigmoid (so the final answer looks like a probability).

The whole left-to-right computation is called the **forward pass**.

In [ ]:
# A tiny 2-layer network forward pass, by hand.
x = np.array([0.5, -1.0, 2.0])     # 3 inputs

# Hidden layer: 4 neurons, each with 3 weights -> shape (4, 3), plus 4 biases.
W1 = np.random.randn(4, 3) * 0.5
b1 = np.zeros(4)

# Output layer: 1 neuron with 4 weights -> shape (1, 4), plus 1 bias.
W2 = np.random.randn(1, 4) * 0.5
b2 = np.zeros(1)

# Forward pass:
z1 = W1 @ x + b1        # weighted sums for the 4 hidden neurons -> shape (4,)
h  = relu(z1)           # activation on the hidden layer -> shape (4,)
z2 = W2 @ h + b2        # weighted sum for the output neuron -> shape (1,)
y  = sigmoid(z2)        # final activation -> a probability-like number

print("hidden pre-activation z1:", np.round(z1, 3))
print("hidden after ReLU  h   :", np.round(h, 3))
print("network output y        :", np.round(y, 4))  # one number in (0, 1)

**What this does:** It runs data through two layers. `W1 @ x` gives all 4 hidden weighted sums in one shot; `relu` bends them; `W2 @ h` combines the hidden values into the final score; `sigmoid` squashes it. The shapes matter: a `(4, 3)` weight matrix times a length-3 input gives 4 outputs — one per hidden neuron. This exact pattern (matrix multiply, add bias, activate, repeat) is what runs inside an LLM, just with far bigger matrices and many more layers.

### ✏️ Exercise

**Task:** Add a **second hidden layer** of 3 neurons (with ReLU) between the existing hidden layer and the output. You'll need new weights `W1b` of shape `(3, 4)` and biases `b1b` of shape `(3,)`, and you must change the output layer's weights `W2` to shape `(1, 3)` so it reads 3 values. Print the final output.

**Hint:** Insert `z1b = W1b @ h + b1b` then `h2 = relu(z1b)`, and feed `h2` into the output layer instead of `h`.

In [ ]:
# Sample solution
W1b = np.random.randn(3, 4) * 0.5   # second hidden layer: 3 neurons, 4 inputs each
b1b = np.zeros(3)
W2  = np.random.randn(1, 3) * 0.5   # output now reads 3 values

z1  = W1 @ x + b1;   h  = relu(z1)    # first hidden layer (4 neurons)
z1b = W1b @ h + b1b; h2 = relu(z1b)   # second hidden layer (3 neurons)
z2  = W2 @ h2 + b2
y   = sigmoid(z2)
print("3-layer network output:", np.round(y, 4))

## 4. Loss: measuring how wrong we are

A network's output is a guess. To improve it we need a **single number that says how bad the guess is** — that's the **loss**. Lower loss = better. Two common ones:

- **Mean Squared Error (MSE)**: average of `(prediction - target)²`. Great when the target is a real number. Squaring makes all errors positive and punishes big misses harder.
- **Binary Cross-Entropy (BCE)**: the go-to loss when the target is **0 or 1** (yes/no). It rewards confident-correct answers and heavily penalizes confident-wrong ones.

Let's implement both.

In [ ]:
# Mean Squared Error: average of (prediction - target) squared.
def mse(pred, target):
    return np.mean((pred - target) ** 2)

# Binary Cross-Entropy: target must be 0 or 1; pred is a probability in (0, 1).
# The tiny "eps" keeps log() from ever seeing exactly 0 (which would be -inf).
def bce(pred, target, eps=1e-9):
    pred = np.clip(pred, eps, 1 - eps)
    return -np.mean(target * np.log(pred) + (1 - target) * np.log(1 - pred))

preds   = np.array([0.9, 0.2, 0.8])   # the network's guesses
targets = np.array([1.0, 0.0, 1.0])   # the correct answers

print("MSE:", round(float(mse(preds, targets)), 4))   # small -> guesses are close
print("BCE:", round(float(bce(preds, targets)), 4))   # small -> confident & correct

# Now make one guess confidently WRONG and watch BCE punish it hard:
bad = np.array([0.9, 0.99, 0.8])      # middle guess 0.99 but target is 0
print("BCE with a confident mistake:", round(float(bce(bad, targets)), 4))  # much bigger

**What this does:** It computes two error scores. With good guesses both losses are small. The last line shows BCE's personality: guessing 0.99 when the true answer is 0 produces a **large** loss, because cross-entropy strongly discourages being confidently wrong. During training we'll keep watching a loss like this and try to drive it down.

### ✏️ Exercise

**Task:** Compute the MSE between `pred = [3.0, 5.0, 2.0]` and `target = [2.5, 5.0, 0.0]` **by hand** (using only `+`, `-`, `*`, and division), then confirm it equals `mse(pred, target)`.

**Hint:** The three squared errors are `0.25`, `0.0`, and `4.0`. Average them by adding and dividing by 3.

In [ ]:
# Sample solution
pred   = np.array([3.0, 5.0, 2.0])
target = np.array([2.5, 5.0, 0.0])

by_hand = (0.25 + 0.0 + 4.0) / 3        # = 4.25 / 3
print("by hand:", round(by_hand, 4))            # -> 1.4167
print("mse():  ", round(float(mse(pred, target)), 4))  # -> 1.4167 (matches)

## 5. Gradients & backpropagation, intuitively

We can measure the loss — now how do we **shrink** it? We adjust the weights. But which direction should each weight move, and how much?

**Gradient = "which way reduces the loss, and how steeply."** For each weight, the gradient answers: *"if I nudge this weight up a little, does the loss go up or down, and by how much?"* If nudging a weight **up** makes the loss go **up**, we should move that weight **down** — i.e., move *opposite* to the gradient. That single rule is **gradient descent**:

`new_weight = old_weight - learning_rate * gradient`

The **learning rate** is a small number (like 0.1) controlling step size: too big and you overshoot, too small and you crawl.

### Backpropagation = assigning blame backward

A network has many layers, so when the final output is wrong, **which weights are to blame?** Backpropagation answers this by passing the blame **backward**, layer by layer, from the output toward the input.

Think of an assembly line. The final product is faulty (high loss). The last station gets blamed first; it then says "but I was handed bad parts," and passes a share of the blame to the station before it, which passes blame further back, and so on. Each station ends up with its own share of responsibility — that share *is* its gradient.

The "**chain rule**" from calculus is just the bookkeeping for this blame-passing: the blame on an early weight = (how much it affected the next thing) × (how much that thing affected the loss). It's **multiply the local effects along the chain**. No need to memorize calculus formulas here — just remember: *backprop multiplies "local blame" values backward through the layers.*

Let's make it concrete with a single neuron, where we can write the gradient out directly.

In [ ]:
# One neuron, one example. We'll compute the gradient of MSE w.r.t. w and b
# by passing blame backward through the three steps of the neuron.
x = np.array([1.5, -2.0])    # inputs
w = np.array([0.0, 0.0])     # start the weights at zero
b = 0.0
target = 1.0                 # the correct answer for this example

# ---- Forward pass (compute the prediction and the loss) ----
z    = np.dot(w, x) + b      # step 1+2: weighted sum
pred = sigmoid(z)            # step 3: activation
loss = (pred - target) ** 2  # MSE for a single example

# ---- Backward pass (assign blame back to w and b) ----
# Blame flows: loss <- pred <- z <- (w, b). We multiply each local effect.
d_loss_d_pred = 2 * (pred - target)        # how loss changes as pred changes
d_pred_d_z    = pred * (1 - pred)          # how pred changes as z changes (sigmoid slope)
d_loss_d_z    = d_loss_d_pred * d_pred_d_z # chain them: multiply the blames

# z = w·x + b, so z's blame splits onto each weight in proportion to its input x,
# and the bias gets the blame directly.
grad_w = d_loss_d_z * x      # gradient for each weight
grad_b = d_loss_d_z          # gradient for the bias

print("loss   :", round(float(loss), 4))
print("grad_w :", np.round(grad_w, 4))   # which way (and how hard) to move each weight
print("grad_b :", round(float(grad_b), 4))

**What this does:** It computes gradients by hand for one neuron. The forward pass produces a prediction and a loss. The backward pass multiplies three "local blames" — how the loss reacts to `pred`, how `pred` reacts to `z`, and how `z` reacts to each weight (which is just the input `x`). The results `grad_w` and `grad_b` tell us exactly which direction to move each parameter to reduce the loss. In the next section we'll actually take those steps, over and over, and watch the loss fall.

### ✏️ Exercise

**Task:** Using the variables above, apply **one** gradient-descent step to `w` and `b` with a learning rate of `0.5`, then recompute the loss. Did it go down?

**Hint:** `w = w - 0.5 * grad_w` and `b = b - 0.5 * grad_b`. Then redo the forward pass to get the new loss.

In [ ]:
# Sample solution
lr = 0.5
w = w - lr * grad_w
b = b - lr * grad_b

new_pred = sigmoid(np.dot(w, x) + b)
new_loss = (new_pred - target) ** 2
print("loss before:", round(float(loss), 4))
print("loss after :", round(float(new_loss), 4))   # should be smaller

## 6. Hands-on: train a neuron and watch the loss fall

Now we put it all together. We'll train a single sigmoid neuron to **classify** a tiny toy dataset, using manual gradient descent — the exact same loop that (scaled up enormously) trains and fine-tunes LLMs.

**The toy task:** given two numbers per example, output 1 if their sum is positive, else 0. We make 8 examples by hand.

In [ ]:
# Tiny toy dataset: 8 examples, 2 features each.
X = np.array([
    [ 2.0,  1.0],
    [ 1.0,  3.0],
    [ 3.0,  2.0],
    [ 0.5,  1.5],
    [-2.0, -1.0],
    [-1.0, -3.0],
    [-3.0, -1.5],
    [-0.5, -2.0],
])
# Label = 1 if the two features sum to a positive number, else 0.
y = np.array([1, 1, 1, 1, 0, 0, 0, 0], dtype=float)

print("X shape:", X.shape, "| y:", y)   # -> X shape: (8, 2) | y: [1. 1. 1. 1. 0. 0. 0. 0.]

In [ ]:
# Initialize the neuron's parameters and a learning rate.
w = np.random.randn(2) * 0.1   # 2 weights, small random start
b = 0.0
lr = 0.5                       # learning rate (step size)
n  = len(y)                    # number of examples

losses = []                    # we'll record the loss at every step to plot later

for step in range(60):
    # ---- Forward pass over ALL examples at once ----
    z    = X @ w + b           # weighted sums -> shape (8,)
    pred = sigmoid(z)          # probabilities -> shape (8,)
    loss = bce(pred, y)        # one number: average binary cross-entropy
    losses.append(loss)

    # ---- Backward pass (gradients, averaged over the 8 examples) ----
    # For sigmoid + BCE the blame on z simplifies neatly to (pred - y).
    d_z    = (pred - y) / n            # blame on each z, averaged
    grad_w = X.T @ d_z                 # split blame onto each weight via its inputs
    grad_b = np.sum(d_z)               # bias soaks up the blame directly

    # ---- Gradient-descent step: move opposite the gradient ----
    w = w - lr * grad_w
    b = b - lr * grad_b

    if step % 10 == 0:
        print(f"step {step:2d}  loss = {loss:.4f}")
# -> loss should DECREASE as step increases, e.g. 0.7-ish down toward ~0.1

**What this does:** This is a complete training loop. Each iteration: (1) **forward** — predict on all 8 examples and measure the BCE loss; (2) **backward** — compute how to blame each weight (`X.T @ d_z`) and the bias; (3) **update** — step each parameter opposite its gradient. Repeating this 60 times steadily lowers the loss. For the sigmoid+BCE pairing, the blame on `z` happens to simplify to exactly `(pred - y)`, which is why the gradient code is so short.

In [ ]:
# Plot the loss curve to SEE it go down.
plt.figure(figsize=(7, 4))
plt.plot(losses, marker=".")
plt.title("Training loss going down over steps")
plt.xlabel("training step"); plt.ylabel("BCE loss")
plt.grid(True, alpha=0.3)
plt.show()
# -> A line that starts higher and slopes downward, flattening as it learns.

In [ ]:
# Did it actually learn? Check predictions vs. the true labels.
final_pred = sigmoid(X @ w + b)
print("learned weights:", np.round(w, 3), "| bias:", round(float(b), 3))
print("predicted prob :", np.round(final_pred, 2))
print("rounded guess  :", (final_pred > 0.5).astype(int))
print("true labels    :", y.astype(int))
# -> rounded guesses should match the true labels (all, or nearly all, correct)

**What this does:** It uses the trained weights to predict on the dataset and compares to the truth. The probabilities should be high (near 1) for the positive-sum examples and low (near 0) for the rest, so the rounded guesses match the labels. You just trained a neuron from scratch — the same loop, scaled up, is how LLMs are trained and fine-tuned.

### ✏️ Exercise

**Task:** Re-run the training loop with a **much larger** learning rate, `lr = 50.0`. Look at the printed losses. Do they fall smoothly, bounce around, or blow up? Then try a **tiny** `lr = 0.001` — does it learn within 60 steps?

**Hint:** Just change the `lr = ...` line and re-run the training cell and the loss-plot cell. Big steps can overshoot and destabilize; tiny steps barely move.

In [ ]:
# Sample solution — wrap training in a function so we can try different rates.
def train(lr, steps=60, seed=0):
    rng = np.random.RandomState(seed)
    w = rng.randn(2) * 0.1
    b = 0.0
    hist = []
    for _ in range(steps):
        pred = sigmoid(X @ w + b)
        hist.append(bce(pred, y))
        d_z = (pred - y) / len(y)
        w = w - lr * (X.T @ d_z)
        b = b - lr * np.sum(d_z)
    return hist

for lr_try in [0.001, 0.5, 50.0]:
    h = train(lr_try)
    print(f"lr={lr_try:<7} first loss={h[0]:.3f}  last loss={h[-1]:.3f}")
# -> lr=0.5 lowers loss nicely; lr=0.001 barely moves; lr=50 may bounce/diverge.

## Common mistakes & how to debug them

- **Forgetting the activation between layers.** Stacking linear layers with no nonlinearity collapses to a single layer (we proved this). If a deep network learns no better than a 1-layer one, check that activations are actually applied.
- **Shape mismatches in `np.dot` / `@`.** The classic error is `shapes (a,b) and (c,d) not aligned`. Print `.shape` of every array. For `W @ x`, the number of columns of `W` must equal the length of `x`.
- **`log(0)` in cross-entropy → `nan` or `-inf`.** If predictions can hit exactly 0 or 1, clip them first (we used `np.clip(pred, eps, 1 - eps)`). Seeing `nan` in your loss almost always means an unclipped log or an exploding learning rate.
- **Learning rate too big or too small.** Loss bouncing or shooting to `nan` → lower the rate. Loss barely moving → raise it. Start around `0.1`–`1.0` for toy problems and adjust.
- **Wrong sign on the update.** Gradient descent **subtracts** the gradient (`w - lr*grad`). Adding it instead makes the loss *climb*. If your loss goes up every step, check this sign.
- **Loss not decreasing at all.** Print the loss every few steps (as we did). A flat or rising curve is your fastest clue that something above is wrong.

## Summary

- A **neuron** = weighted sum of inputs + bias, then an **activation**. In NumPy that's `sigmoid(np.dot(w, x) + b)`.
- **Activations** (sigmoid, tanh, ReLU) add the **nonlinearity** that makes depth meaningful — without them, stacked linear layers collapse into one.
- A **network** is layers of neurons; the **forward pass** is repeated "matrix-multiply → add bias → activate."
- A **loss** (MSE, BCE) turns "how wrong are we?" into a single number to minimize.
- A **gradient** points the way to reduce the loss; **backpropagation** assigns blame backward through the layers (the chain rule = multiplying local blames). **Gradient descent** then steps each weight *opposite* its gradient.
- You **trained a neuron** by hand and watched the loss fall — the exact loop, scaled up, behind every LLM.

The big connection: **an LLM is a giant neural network, and fine-tuning is continuing to nudge its weights with gradient descent on your data.** Everything here is that process in miniature.

## What to learn next

Up next: **`04_pytorch_fundamentals.ipynb`**. We've done everything by hand in NumPy so you can *see* the mechanics. PyTorch automates the painful part — it computes gradients (backpropagation) for you automatically and runs fast on GPUs. In the next notebook you'll meet **tensors** (NumPy-like arrays that track gradients), `autograd` (so you never hand-derive a gradient again), and the standard PyTorch training loop — recognizing it as the same forward → loss → backward → update cycle you just built here.